In [47]:
import json  # Parse BigQuery's JSON response into Python records.
from io import StringIO  # Let pandas read BigQuery command output held in memory.
import os  # Set a temporary Matplotlib configuration directory below.
import subprocess  # Run authenticated BigQuery CLI commands from Python.
import tempfile  # Find a safe temporary directory for Matplotlib files.
from pathlib import Path  # Build filesystem paths in an OS-independent way.

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "amazon_books_matplotlib"),
)

import matplotlib.pyplot as plt  # Create plots later in the notebook.
import numpy as np  # Numerical arrays and calculations.
import pandas as pd  # Load, inspect, and transform table-shaped data.
from IPython.display import display

PROJECT = "wagon-bootcamp-501605"
DATASET = "amazon_books_2023"
TABLE = "business_money_usable_tocs_9034"
LOCATION = "asia-northeast1"
RANDOM_STATE = 42
FULL_TABLE = f"{PROJECT}.{DATASET}.{TABLE}"
print(FULL_TABLE)

wagon-bootcamp-501605.amazon_books_2023.business_money_usable_tocs_9034


# Prepare the usable TOCs

The BigQuery table is already a curated source: every record has an exact-ISBN Open Library match and at least three extracted TOC entries. We keep the raw fields unchanged and build a separate, model-ready table.

Following the data-preparation lecture, we audit duplicates and missing data before transforming text. The unit of analysis is `parent_asin`, not ISBN: 39 canonical ISBNs occur in more than one Amazon record.

In [48]:
# Load every source column first. JSON preserves nested and repeated fields
# that BigQuery cannot serialize in CSV.
query = f"""
SELECT *
FROM `{FULL_TABLE}`
ORDER BY parent_asin
"""

command = [
    "bq",
    f"--location={LOCATION}",
    "query",
    "--use_legacy_sql=false",
    "--format=json",
    "--max_rows=10000",
    query,
]
print("1/7 -- Full-table BigQuery command is ready.")

1/7 -- Full-table BigQuery command is ready.


In [49]:
# Run the full-table query. Its JSON output can preserve nested columns. (50 seconds)
result = subprocess.run(command, capture_output=True, text=True, check=True)
print(f"2/7 -- Downloaded {len(result.stdout):,} characters of JSON.")

2/7 -- Downloaded 216,668,245 characters of JSON.


In [50]:
# Convert the JSON response into one pandas row per book. (9,034 rows x 38 columns.)
source_books = pd.DataFrame(json.loads(result.stdout))
print(f"3/7 -- Created source_books: {source_books.shape[0]:,} rows x {source_books.shape[1]:,} columns.")

3/7 -- Created source_books: 9,034 rows x 38 columns.


In [51]:
# Store identifier-like values as text so leading zeros and ISBN-10's final X are never lost.
identifier_columns = [
    "parent_asin", "amazon_isbn_10", "amazon_isbn_13",
    "canonical_isbn_13", "ol_edition_key",
]
for column in identifier_columns:
    source_books[column] = source_books[column].astype("string")
print("4/7 -- Identifier dtypes:")
print(source_books[identifier_columns].dtypes)

4/7 -- Identifier dtypes:
parent_asin          string[python]
amazon_isbn_10       string[python]
amazon_isbn_13       string[python]
canonical_isbn_13    string[python]
ol_edition_key       string[python]
dtype: object


In [52]:
# CHECKPOINT -- before 5/7: we have preserved the original book-level table.
print("Before TOC flattening")
print(f"- source_books contains {len(source_books):,} books and {len(source_books.columns):,} original columns.")
print("- Each row is one Amazon book; nested fields such as toc_entries are still intact.")
print("- Next we create a separate entry-level view only to validate and clean TOC text.")
display(source_books[["parent_asin", "title", "toc_entry_count", "toc_entries"]].head(3))

Before TOC flattening
- source_books contains 9,034 books and 38 original columns.
- Each row is one Amazon book; nested fields such as toc_entries are still intact.
- Next we create a separate entry-level view only to validate and clean TOC text.


,parent_asin,title,toc_entry_count,toc_entries
0,000216132X,"The wheels of commerce, vol.2: civilisation an...",3,"[{'level': None, 'page': None, 'sequence': '1'..."
1,0007519532,Will there be Donuts?: Better Business One Mee...,5,"[{'level': '0', 'page': None, 'sequence': '1',..."
2,0021057117,California Mathematics Grade 4 (Student Editio...,20,"[{'level': '0', 'page': None, 'sequence': '1',..."


In [53]:
# source_books has one row per book and preserves every original column.
# This separate query does not append to or replace source_books. It creates
# toc_items: one row per nested TOC entry, which lets us check entry order,
# blanks, and duplicates before joining only cleaned toc_text back to books.
#
# Before flattening, one book stores its TOC as a list inside one cell:
# parent_asin = "000216132X"
# toc_entries = [{"sequence": 1, "text": "v. 1. The structures..."},
#                {"sequence": 2, "text": "v. 2. The wheels..."},
#                {"sequence": 3, "text": "v. 3. The perspective..."}]
# After flattening, those become three rows with parent_asin, toc_sequence,
# and toc_text_raw. The displays below use a real book from this table.
example_book = source_books.iloc[0]
example_parent_asin = example_book["parent_asin"]
print(f"Nested TOC -- real book: {example_parent_asin} | {example_book['title']}")
display(pd.DataFrame(example_book["toc_entries"])[["sequence", "text"]])
toc_query = f"""
SELECT parent_asin, raw_toc_item_count, toc_entry_count,
  entry.sequence AS toc_sequence, entry.text AS toc_text_raw,
  entry.level AS toc_level, entry.page AS toc_page
FROM `{FULL_TABLE}`
CROSS JOIN UNNEST(toc_entries) AS entry
ORDER BY parent_asin, toc_sequence
"""
toc_command = [
    "bq", f"--location={LOCATION}", "query",
    "--use_legacy_sql=false", "--format=csv", "--max_rows=1000000", toc_query,
]
print("5/7 -- Flattened TOC query is ready.")

Nested TOC -- real book: 000216132X | The wheels of commerce, vol.2: civilisation and capitalism 15th-18th


,sequence,text
0,1,v. 1. The structures of everyday life : the li...
1,2,v. 2. The wheels of commerce
2,3,v. 3. The perspective of the world.


5/7 -- Flattened TOC query is ready.


In [54]:
# Run the simple flat query and read its CSV response into pandas.
toc_result = subprocess.run(toc_command, capture_output=True, text=True, check=True)
toc_items = pd.read_csv(StringIO(toc_result.stdout), dtype={"parent_asin": "string"})
print(f"6/7 -- Loaded {len(toc_items):,} TOC entries for {toc_items['parent_asin'].nunique():,} books.")

6/7 -- Loaded 141,495 TOC entries for 9,034 books.


In [55]:
# Inspect one book without truncating its title or TOC text.
BOOK_TO_INSPECT = "000216132X"
book_record = source_books.loc[source_books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
book_toc = toc_items.loc[
    toc_items["parent_asin"] == BOOK_TO_INSPECT,
    ["toc_sequence", "toc_text_raw", "toc_level", "toc_page"],
].sort_values("toc_sequence")

print(f"Amazon title: {book_record['title']}")
print(f"Open Library edition: {book_record['ol_edition_key']}")
print(f"Declared TOC entries: {book_record['toc_entry_count']}")
print("\nFull table of contents:")
for entry in book_toc.itertuples(index=False):
    print(f"{entry.toc_sequence}. {entry.toc_text_raw}")

Amazon title: The wheels of commerce, vol.2: civilisation and capitalism 15th-18th
Open Library edition: /books/OL22124972M
Declared TOC entries: 3

Full table of contents:
1. v. 1. The structures of everyday life : the limits of the possible
2. v. 2. The wheels of commerce
3. v. 3. The perspective of the world.


In [56]:
# CHECKPOINT -- after 5/7 and 6/7: the original table is still separate.
print("After TOC flattening")
print(f"- source_books is unchanged: {len(source_books):,} books x {len(source_books.columns):,} columns.")
print(f"- toc_items is a separate working view: {len(toc_items):,} TOC-entry rows for {toc_items['parent_asin'].nunique():,} books.")
print("- Next we audit TOC entries, clean their text, then join one cleaned toc_text column back to the full book table.")
print(f"Flattened TOC -- same real book: {example_parent_asin}")
display(
    toc_items.loc[toc_items["parent_asin"] == example_parent_asin,
                  ["parent_asin", "toc_sequence", "toc_text_raw"]]
)

After TOC flattening
- source_books is unchanged: 9,034 books x 38 columns.
- toc_items is a separate working view: 141,495 TOC-entry rows for 9,034 books.
- Next we audit TOC entries, clean their text, then join one cleaned toc_text column back to the full book table.
Flattened TOC -- same real book: 000216132X


,parent_asin,toc_sequence,toc_text_raw
0,000216132X,1,v. 1. The structures of everyday life : the li...
1,000216132X,2,v. 2. The wheels of commerce
2,000216132X,3,v. 3. The perspective of the world.


In [57]:
# Inspect three full source records. random_state makes the sample repeatable.
sampled_books = source_books.sample(n=3, random_state=RANDOM_STATE)
for _, record in sampled_books.iterrows():
    print(json.dumps(record.to_dict(), indent=2, ensure_ascii=False, default=str))
    print("\n" + "-" * 100 + "\n")

{
  "amazon_business_money_record_number": "26513",
  "amazon_isbn_10": "0631228306",
  "amazon_isbn_13": "9780631228301",
  "amazon_raw_record_json": "{\"main_category\": \"Books\", \"title\": \"Carbonell Museum Studies: An Anthology of Contexts\", \"subtitle\": \"1st Edition\", \"author\": null, \"average_rating\": 4.6, \"rating_number\": 8, \"features\": [\"Museum Studies: An Anthology of Contexts\", \"provides a comprehensive interdisciplinary collection of approaches to museums and their relation to history, culture, philosophy and their adoring or combative publics.\", \"Brings together for the first time a wide array of texts that mix contemporary analysis with historical documentation\", \"Brings together for the first time a wide array of texts that mix contemporary analysis with historical documentation\", \"Includes five sections that highlight central themes in museum studies: issue-oriented contexts in museology; states of \\\"nature\\\"; the status of nations; history, me

## 1. Audit duplicates and missing values

Do not remove a book merely because it shares an ISBN or an Open Library edition with another Amazon record. First inspect the fields below; only exact duplicate `parent_asin`/sequence pairs or blank TOC text would be invalid at this grain.

In [58]:
book_counts = toc_items.groupby("parent_asin").agg(
    observed_entries=("toc_sequence", "size"),
    declared_entries=("toc_entry_count", "first"),
)

quality = pd.Series(
    {
        "TOC-entry rows": len(toc_items),
        "books (parent_asin)": toc_items["parent_asin"].nunique(),
        "duplicate parent_asin / sequence pairs": toc_items.duplicated(["parent_asin", "toc_sequence"]).sum(),
        "blank TOC entries": toc_items["toc_text_raw"].fillna("").str.strip().eq("").sum(),
        "books below the 3-entry rule": (book_counts["declared_entries"] < 3).sum(),
        "books whose observed and declared entry counts differ": (book_counts["observed_entries"] != book_counts["declared_entries"]).sum(),
        "books where raw and normalized TOC counts differ": (
            toc_items.drop_duplicates("parent_asin")["raw_toc_item_count"]
            != toc_items.drop_duplicates("parent_asin")["toc_entry_count"]
        ).sum(),
    },
    name="count",
).to_frame()
display(quality)
display(toc_items.isna().mean().sort_values(ascending=False).rename("missing_share").to_frame())

,count
TOC-entry rows,141495
books (parent_asin),9034
duplicate parent_asin / sequence pairs,0
blank TOC entries,0
books below the 3-entry rule,0
books whose observed and declared entry counts differ,0
books where raw and normalized TOC counts differ,17


,missing_share
toc_page,0.954952
toc_level,0.073289
parent_asin,0.000000
raw_toc_item_count,0.000000
toc_entry_count,0.000000
toc_sequence,0.000000
toc_text_raw,0.000000


## 2. Create book-level TOC text (without changing content)

No content transformation is applied yet: TOC numbers, punctuation, capitalization, page numbers, and hierarchy may be meaningful. We validate the raw entries and add one convenient book-level `toc_text` column while keeping every raw field available.

In [59]:
# 2.1 -- Validate the raw TOC structure before using it.
duplicate_pairs = toc_items.duplicated(["parent_asin", "toc_sequence"]).sum()
blank_raw_entries = toc_items["toc_text_raw"].isna().sum() + toc_items["toc_text_raw"].eq("").sum()
assert duplicate_pairs == 0
assert blank_raw_entries == 0
print(f"2.1 -- Validation passed: {duplicate_pairs} duplicate book/sequence pairs; {blank_raw_entries} blank raw entries.")

2.1 -- Validation passed: 0 duplicate book/sequence pairs; 0 blank raw entries.


In [60]:
# 2.2 -- Inspect a raw book, then reassemble its ordered raw entries into toc_text.
raw_book = source_books.loc[source_books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print("Raw source record (before any content transformation):")
print(f"Title: {raw_book['title']}")
print(f"Amazon ISBN-10: {raw_book['amazon_isbn_10']}")
print(f"Amazon ISBN-13: {raw_book['amazon_isbn_13']}")
print(f"Canonical ISBN-13: {raw_book['canonical_isbn_13']}")
print(f"Parent ASIN: {raw_book['parent_asin']}")
print("\nOriginal nested toc_entries:")
print(json.dumps(raw_book["toc_entries"], indent=2, ensure_ascii=False))

toc_text_by_book = (
    toc_items.sort_values(["parent_asin", "toc_sequence"])
    .groupby("parent_asin")["toc_text_raw"]
    .agg("\n".join)
)
toc_text_by_book.name = "toc_text"
print(f"2.2 -- Created one ordered TOC text value for {len(toc_text_by_book):,} books.")
print(f"\nTOC for {BOOK_TO_INSPECT}:\n{toc_text_by_book.loc[BOOK_TO_INSPECT]}")

Raw source record (before any content transformation):
Title: The wheels of commerce, vol.2: civilisation and capitalism 15th-18th
Amazon ISBN-10: 000216132X
Amazon ISBN-13: 9780002161329
Canonical ISBN-13: 9780002161329
Parent ASIN: 000216132X

Original nested toc_entries:
[
  {
    "level": null,
    "page": null,
    "sequence": "1",
    "source_key": "value",
    "text": "v. 1. The structures of everyday life : the limits of the possible"
  },
  {
    "level": null,
    "page": null,
    "sequence": "2",
    "source_key": "value",
    "text": "v. 2. The wheels of commerce"
  },
  {
    "level": null,
    "page": null,
    "sequence": "3",
    "source_key": "value",
    "text": "v. 3. The perspective of the world."
  }
]
2.2 -- Created one ordered TOC text value for 9,034 books.

TOC for 000216132X:
v. 1. The structures of everyday life : the limits of the possible
v. 2. The wheels of commerce
v. 3. The perspective of the world.


## 3. strip(), lower case (when embedding words upcase/lowercase matter)


# 3 -- Optional variant: trim only the outside whitespace and lowercase TOC text.
# The unchanged books['toc_text'] remains our baseline and is never overwritten.
books = source_books.copy().join(toc_text_by_book, on="parent_asin")
books["toc_text_lower"] = books["toc_text"].str.strip().str.lower()
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print("3 -- Optional lowercase variant created; baseline toc_text is unchanged.")
print(f"Baseline:  {example['toc_text']!r}")
print(f"Lowercase: {example['toc_text_lower']!r}")

## 4. dealing with numbers, punctuation, and symbols

In [ ]:
# 4 -- Optional variant: remove digits, punctuation, and symbols from a copy.
# This is deliberately not applied to toc_text because TOC markers such as 1. and Part II may matter.
import re
books = source_books.copy().join(toc_text_by_book, on="parent_asin")
books["toc_text_letters_only"] = (
    books["toc_text"].str.replace(r"\b[vV]\b\.?", "", regex=True)  # remove v, v.
        .str.replace(r"[^a-zA-Z\s]", " ", regex=True)  # change number,
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
)
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print("4 -- Optional letters-only variant created; baseline toc_text is unchanged.")
print(f"Baseline:     {example['toc_text']!r}")
print(f"Letters only: {example['toc_text_letters_only']!r}")

4 -- Optional letters-only variant created; baseline toc_text is unchanged.
Baseline:     'v. 1. The structures of everyday life : the limits of the possible\nv. 2. The wheels of commerce\nv. 3. The perspective of the world.'
Letters only: 'The structures of everyday life the limits of the possible The wheels of commerce The perspective of the world'


## 5. splitting

# 5 -- Optional structural view: split the multiline TOC into its original entries.
# This creates a list for analysis; it does not change toc_text.
# pandas has no .str.splitlines(); our book-level toc_text uses newline separators.
books["toc_lines"] = books["toc_text"].str.split("\n")
books["toc_line_count"] = books["toc_lines"].str.len()
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print(f"5 -- Split TOC into {example['toc_line_count']} lines for {BOOK_TO_INSPECT}.")
for position, line in enumerate(example["toc_lines"], start=1):
    print(f"{position}. {line}")

## 6. tokenizing

# 6 -- Optional token view: extract word-like tokens into a separate list.
# Most vectorizers/tokenizers do this themselves, so toc_text stays the baseline input.
books["toc_tokens"] = books["toc_text"].str.findall(r"\b\w+\b")
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print(f"6 -- Extracted {len(example['toc_tokens'])} tokens for {BOOK_TO_INSPECT}.")
print(example["toc_tokens"][:40])

## 7. removing "stopwords"

# 7 -- Optional token variant: remove English stopwords from a copy of the token list.
# Keep this experimental: words such as 'part' or 'chapter' may matter for TOCs.
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
books["toc_tokens_no_stopwords"] = books["toc_tokens"].map(
    lambda tokens: [token for token in tokens if token.casefold() not in ENGLISH_STOP_WORDS]
)
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print(f"7 -- Tokens: {len(example['toc_tokens'])}; after stopword removal: {len(example['toc_tokens_no_stopwords'])}.")
print(example["toc_tokens_no_stopwords"][:40])

## 8. lemmatizing

# 8 -- Optional token variant: apply simple noun lemmatization to a copy.
# This requires nltk in the current notebook kernel; all earlier steps work without it.
try:
    from nltk.stem import WordNetLemmatizer
except ModuleNotFoundError:
    print("8 -- Skipped: nltk is not installed in this notebook kernel.")
    print("Run `%pip install -e .` from the project folder, restart the kernel, then rerun this optional cell.")
else:
    lemmatizer = WordNetLemmatizer()
    try:
        books["toc_tokens_lemmatized"] = books["toc_tokens"].map(
            lambda tokens: [lemmatizer.lemmatize(token.casefold()) for token in tokens]
        )
    except LookupError:
        print("8 -- Skipped: the NLTK WordNet data is not installed.")
        print("Run `import nltk; nltk.download('wordnet')`, then rerun this optional cell.")
    else:
        example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
        print("8 -- Original tokens versus optional noun-lemmatized tokens:")
        print(f"Original:    {example['toc_tokens'][:40]}")
        print(f"Lemmatized:  {example['toc_tokens_lemmatized'][:40]}")

print(f"Intermediate result: {len(books):,} rows x {len(books.columns):,} columns.")
print(f"Columns: {books.columns.tolist()}")

In [64]:
difficulty_query = """
SELECT *
FROM `wagon-bootcamp-501605.amazon_books_2023.difficulty_classifications_8995`
"""
difficulty_command = [
    "bq", "--location=asia-northeast1", "query",
    "--use_legacy_sql=false", "--format=json", "--max_rows=100000",
    difficulty_query,
]
difficulty_result = subprocess.run(difficulty_command, capture_output=True, text=True, check=True)
difficulty = pd.DataFrame(json.loads(difficulty_result.stdout))
print(f"difficulty_classifications_8995: {difficulty.shape[0]:,} rows x {difficulty.shape[1]:,} columns.")
print(f"Columns: {difficulty.columns.tolist()}")

assert "parent_asin" in difficulty.columns, "No parent_asin found -- check the real join key!"
difficulty["parent_asin"] = difficulty["parent_asin"].astype("string")

difficulty_classifications_8995: 8,995 rows x 13 columns.
Columns: ['canonical_isbn_13', 'confidence', 'difficulty_level', 'difficulty_score', 'evidence', 'input_tokens', 'model', 'output_tokens', 'parent_asin', 'reason', 'result_source', 'rubric_version', 'title']


In [73]:
#books = books.merge(
#    difficulty[["parent_asin", "difficulty_score"]],
#    on="parent_asin",
#    how="left",
#)

print(f"books: {len(books):,} rows x {len(books.columns):,} columns.")
print(f"Missing difficulty_score: {books['difficulty_score'].isna().sum():,}")
print(books["difficulty_score"].value_counts(dropna=False).sort_index())
display(books[["parent_asin", "title", "toc_text_letters_only", "difficulty_score"]].head(3))

books: 9,034 rows x 43 columns.
Missing difficulty_score: 39
difficulty_score
1      1622
2      4135
3      2431
4       588
5       219
NaN      39
Name: count, dtype: int64


,parent_asin,title,toc_text_letters_only,difficulty_score
0,000216132X,"The wheels of commerce, vol.2: civilisation an...",The structures of everyday life the limits of ...,5
1,0007519532,Will there be Donuts?: Better Business One Mee...,Nearly meeting Really meeting The anatomy of m...,2
2,0021057117,California Mathematics Grade 4 (Student Editio...,Student edition Teacher edition Teacher editio...,1


In [74]:
def extract_main_image_url(raw_json):
    variants = json.loads(raw_json) if isinstance(raw_json, str) else raw_json
    if not variants:
        return None
    for variant in variants:
        if variant.get("variant") == "MAIN" and variant.get("large"):
            return variant["large"]
    return variants[0].get("large")


def extract_leaf_category(raw_categories):
    categories_list = json.loads(raw_categories) if isinstance(raw_categories, str) else raw_categories
    if isinstance(categories_list, list) and len(categories_list) > 0:
        return categories_list[-1]
    return None


books["cover_image_url"] = books["images_json"].map(extract_main_image_url)
books["category"] = books["categories"].map(extract_leaf_category)
books["ISBN_10"] = books["amazon_isbn_10"]

print(f"Missing cover_image_url: {books['cover_image_url'].isna().sum():,}")
print(f"Missing category: {books['category'].isna().sum():,}")
display(books[["parent_asin", "title", "cover_image_url", "category", "ISBN_10", "difficulty_score"]].head(3))

Missing cover_image_url: 1,916
Missing category: 0


,parent_asin,title,cover_image_url,category,ISBN_10,difficulty_score
0,000216132X,"The wheels of commerce, vol.2: civilisation an...",None,Economics,000216132X,5
1,0007519532,Will there be Donuts?: Better Business One Mee...,None,Management & Leadership,0007519532,2
2,0021057117,California Mathematics Grade 4 (Student Editio...,https://m.media-amazon.com/images/I/51MwPsn3wq...,Management & Leadership,0021057117,1


In [115]:
difficulty

,canonical_isbn_13,confidence,difficulty_level,difficulty_score,evidence,input_tokens,model,output_tokens,parent_asin,reason,result_source,rubric_version,title
0,9781422128848,high,beginner,2,What is a budget? | Types of budgets,475,gpt-5.6-luna,58,1422128849,This practical guide introduces budgeting conc...,full_9034,pilot-v1,Preparing a Budget (Pocket Mentor)
1,9780060889579,high,beginner,1,fans and newcomers alike | mix smart thinking ...,1225,gpt-5.6-luna,60,0060889578,"Popular, accessible economics storytelling int...",full_9034,pilot-v1,"Super Freakonomics: Global Cooling, Patriotic ..."
2,9781422158784,high,beginner,1,"Quick, practical management advice | easy-to-r...",754,gpt-5.6-luna,60,1422158780,This concise guide targets managers seeking im...,full_9034,pilot-v1,Management Tips: From Harvard Business Review
3,9781541773677,high,beginner,2,lucid analysis | accessible framework,1279,gpt-5.6-luna,60,1541773675,"An accessible primer explains trust, technolog...",full_9034,pilot-v1,Who Can You Trust?: How Technology Brought Us ...
4,9781590791523,high,beginner,2,The case for communication | By focusing on th...,709,gpt-5.6-luna,60,1590791525,This leadership communication guide presents f...,full_9034,pilot-v1,The Leader's Voice
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8990,9780470721421,high,advanced,4,Written for Operational Research and Managemen...,915,gpt-5.6-luna,312,0470721421,A specialist management science text for OR st...,full_9034,pilot-v1,Tools for Thinking: Modelling in Management Sc...
8991,9789813206731,medium,beginner,2,Readership: This book is for anyone interested...,1084,gpt-5.6-luna,325,981320673X,The book introduces economic and business-scho...,full_9034,pilot-v1,Seeking Adam Smith: Finding The Shadow Curricu...
8992,9780199919758,high,intermediate,3,"""128 closely argued pages"" | ""intense philosop...",1088,gpt-5.6-luna,347,0199919755,A concise but demanding philosophical critique...,full_9034,pilot-v1,Mind & Cosmos: Why the Materialist Neo-Darwini...
8993,9780415994224,high,intermediate,3,"""step-by-step unfolding of the strategic campa...",850,gpt-5.6-luna,370,0415994225,"This is a structured, practice-oriented textbo...",full_9034,pilot-v1,Strategic Planning for Public Relations


import re
import subprocess
import sys

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Make sure nltk is installed in THIS kernel (the one running this notebook),
# instead of asking for a manual pip install in a separate step.
try:
    import nltk
except ModuleNotFoundError:
    print("nltk is not installed in this kernel -- installing it now...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nltk"], check=True)
    import nltk

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
try:
    lemmatizer.lemmatize("test")
except LookupError:
    print("Downloading NLTK wordnet data (one-time)...")
    nltk.download("wordnet")
    nltk.download("omw-1.4")
    lemmatizer.lemmatize("test")  # confirm it works now

LEMMATIZE_AVAILABLE = True
print("Lemmatizer ready:", lemmatizer.lemmatize("books"))


def process_toc_text(text):
    if not isinstance(text, str) or not text:
        return text
    cleaned = text.strip().lower()
    cleaned = re.sub(r"[^\w\s]|\d", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    tokens = re.findall(r"\b\w+\b", cleaned)
    tokens = [token for token in tokens if token not in ENGLISH_STOP_WORDS]
    if LEMMATIZE_AVAILABLE:
        tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)


books["toc_text_processed"] = books["toc_text"].map(process_toc_text)

example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print("Baseline:  ", example["toc_text"])
print("Processed: ", example["toc_text_processed"])

In [78]:
final_columns = [
    "parent_asin", "title", "ISBN_10", "cover_image_url", "category", "toc_text","toc_text_letters_only", "difficulty_score",
]
books_final = books[final_columns].copy()

print(f"books_final: {len(books_final):,} rows x {len(books_final.columns):,} columns.")
display(books_final.head(3))

# Optional: export as CSV, e.g. for model.py
# books_final.to_csv("exercise_data.csv", index=False)

books_final: 9,034 rows x 8 columns.


,parent_asin,title,ISBN_10,cover_image_url,category,toc_text,toc_text_letters_only,difficulty_score
0,000216132X,"The wheels of commerce, vol.2: civilisation an...",000216132X,None,Economics,v. 1. The structures of everyday life : the li...,The structures of everyday life the limits of ...,5
1,0007519532,Will there be Donuts?: Better Business One Mee...,0007519532,None,Management & Leadership,1. Nearly meeting\n2. Really meeting\n3. The a...,Nearly meeting Really meeting The anatomy of m...,2
2,0021057117,California Mathematics Grade 4 (Student Editio...,0021057117,https://m.media-amazon.com/images/I/51MwPsn3wq...,Management & Leadership,"Student edition\nTeacher edition, v. 1\nTeache...",Student edition Teacher edition Teacher editio...,1


In [79]:
books_final

,parent_asin,title,ISBN_10,cover_image_url,category,toc_text,toc_text_letters_only,difficulty_score
0,000216132X,"The wheels of commerce, vol.2: civilisation an...",000216132X,None,Economics,v. 1. The structures of everyday life : the li...,The structures of everyday life the limits of ...,5
1,0007519532,Will there be Donuts?: Better Business One Mee...,0007519532,None,Management & Leadership,1. Nearly meeting\n2. Really meeting\n3. The a...,Nearly meeting Really meeting The anatomy of m...,2
2,0021057117,California Mathematics Grade 4 (Student Editio...,0021057117,https://m.media-amazon.com/images/I/51MwPsn3wq...,Management & Leadership,"Student edition\nTeacher edition, v. 1\nTeache...",Student edition Teacher edition Teacher editio...,1
3,0026441918,"Marketing Essentials, Third Edition",0026441918,None,Marketing & Sales,The world of marketing\nEconomics\nBusiness an...,The world of marketing Economics Business and ...,2
4,0028634160,The Unofficial Guide to Hot Careers,0028634160,None,Job Hunting & Careers,pt. 1.\nWhat is a hot career? -- -- ch. 1.\nIf...,pt What is a hot career ch If you get a job yo...,1
...,...,...,...,...,...,...,...,...
9029,B01FGMT3O0,"California Mathematics: Concepts, Skills, and ...",0078778484,None,Management & Leadership,"Contents of student edition, grade 6: Introdut...",Contents of student edition grade Introdution ...,1
9030,B01I8HJQSS,Unfinished Business Women Men Work Family,0812994566,https://m.media-amazon.com/images/I/41Z4rJEYue...,Business Culture,"""It's such a pity you had to leave Washington""...",It s such a pity you had to leave Washington P...,2
9031,B01N1WAQLB,Information Systems A Manager's Guide to Harne...,0982361815,https://m.media-amazon.com/images/I/41p7TXpxGj...,Management & Leadership,ch. 1. Setting the stage : technology and the ...,ch Setting the stage technology and the modern...,2
9032,B08B7LNDQ8,How to Listen with Intention: The Foundation o...,,https://m.media-amazon.com/images/I/41Au0RhomM...,Management & Leadership,How to Listen with Intention: The Foundation o...,How to Listen with Intention The Foundation of...,1


In [81]:
books_final[books_final['parent_asin'] != books_final['ISBN_10']]

,parent_asin,title,ISBN_10,cover_image_url,category,toc_text,toc_text_letters_only,difficulty_score
8,0029055156,Revolutionizing Product Development: Quantum L...,,None,Marketing & Sales,Competing through development capability\nThe ...,Competing through development capability The c...,3
26,0060840978,How We Got Here: A Slightly Irreverent History...,,https://m.media-amazon.com/images/I/41rIIcqDQC...,Biography & History,Foreword\nLogic and memory\npt. 1. The Industr...,Foreword Logic and memory pt The Industrial Re...,2
35,0061128821,"The Slow Fix: Solve Problems, Work Smarter, an...",,https://m.media-amazon.com/images/I/41Ksk1Px2G...,Management & Leadership,Why the quick fix?\nConfess: the magic of mist...,Why the quick fix Confess the magic of mistake...,2
54,0061626635,Rebound Rules: The Art of Success 2.0,,https://m.media-amazon.com/images/I/51BR7RprZb...,Business Culture,The darkness of doubt\nGaining perspective\nPH...,The darkness of doubt Gaining perspective PHD ...,2
66,0061766089,Change by Design: How Design Thinking Transfor...,,https://m.media-amazon.com/images/I/41-aaVMQaF...,Management & Leadership,"Getting under your skin, or how design thinkin...",Getting under your skin or how design thinking...,2
...,...,...,...,...,...,...,...,...
9029,B01FGMT3O0,"California Mathematics: Concepts, Skills, and ...",0078778484,None,Management & Leadership,"Contents of student edition, grade 6: Introdut...",Contents of student edition grade Introdution ...,1
9030,B01I8HJQSS,Unfinished Business Women Men Work Family,0812994566,https://m.media-amazon.com/images/I/41Z4rJEYue...,Business Culture,"""It's such a pity you had to leave Washington""...",It s such a pity you had to leave Washington P...,2
9031,B01N1WAQLB,Information Systems A Manager's Guide to Harne...,0982361815,https://m.media-amazon.com/images/I/41p7TXpxGj...,Management & Leadership,ch. 1. Setting the stage : technology and the ...,ch Setting the stage technology and the modern...,2
9032,B08B7LNDQ8,How to Listen with Intention: The Foundation o...,,https://m.media-amazon.com/images/I/41Au0RhomM...,Management & Leadership,How to Listen with Intention: The Foundation o...,How to Listen with Intention The Foundation of...,1


In [82]:
books_final[books_final["parent_asin"].str.strip().eq("")]

,parent_asin,title,ISBN_10,cover_image_url,category,toc_text,toc_text_letters_only,difficulty_score


In [85]:
books_final.drop(columns=['ISBN_10'],inplace=True)

In [90]:
books_final.drop(columns=['toc_text'],inplace=True)

In [94]:
books_final.rename(columns={'toc_text_letters_only': 'toc_text'},inplace=True)

In [95]:
books_final[books_final["parent_asin"].duplicated()]

,parent_asin,title,cover_image_url,category,toc_text,difficulty_score


In [ ]:

books_final[books_final["parent_asin"].duplicated(keep=False)].sort_values("parent_asin")

,parent_asin,title,cover_image_url,category,toc_text,difficulty_score


In [97]:
books_final.duplicated().sum()

np.int64(0)

In [118]:
books_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9031 entries, 0 to 9033
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   parent_asin       9031 non-null   string
 1   title             9031 non-null   object
 2   cover_image_url   7115 non-null   object
 3   category          9031 non-null   object
 4   toc_text          9031 non-null   object
 5   difficulty_score  8992 non-null   object
dtypes: object(5), string(1)
memory usage: 493.9+ KB


In [107]:
# 文字数を計算する列を仮作成
toc_length = books_final["toc_text"].fillna("").str.len()

# 10文字以下の短いテキストを確認（空文字を除く）
books_final[(toc_length > 0) & (toc_length < 10)][["parent_asin", "toc_text"]]


,parent_asin,toc_text
164,0070115435,xi
3097,0679404694,xi
7121,1585425524,xiii xv


In [ ]:

books_final = books_final.drop(index=[164, 3097, 7121])

In [113]:
books_final["toc_text"].fillna("").astype(str)

0       The structures of everyday life the limits of ...
1       Nearly meeting Really meeting The anatomy of m...
2       Student edition Teacher edition Teacher editio...
3       The world of marketing Economics Business and ...
4       pt What is a hot career ch If you get a job yo...
                              ...                        
9029    Contents of student edition grade Introdution ...
9030    It s such a pity you had to leave Washington P...
9031    ch Setting the stage technology and the modern...
9032    How to Listen with Intention The Foundation of...
9033    Acknowledgments vi Introduction Part War Part ...
Name: toc_text, Length: 9031, dtype: object

In [114]:
books_final.to_csv("final_data.csv", index=False)

In [119]:
!pwd

/Users/kanon/Downloads
